In [ ]:
!pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas scikit-learn

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import xml.etree.ElementTree as ET

from google.colab import drive
from datasets import Dataset
from sklearn.model_selection import train_test_split

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# insert your paths here if you are running locally, or use the  paths accordingly if you are using Colab
DATA_DIR = "/content/drive/MyDrive/Deep Learning Mid-Term/dl-spring-2026-svg-generation"
OUTPUT_DIR = "/content/drive/MyDrive/Deep Learning Mid-Term/Model_Output"

CONFIG = {
    "model_name": "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
    "max_seq_length": 1536,
    "lora_r": 16,
    "lora_alpha": 32,
    "learning_rate": 2e-4,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "eval_size": 0.05,
    "output_dir": OUTPUT_DIR,
}

In [ ]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

print(train_df.shape, test_df.shape)
train_df.head()

In [ ]:
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", "polygon", "defs", "use", "symbol", "clipPath", "mask", "linearGradient", "radialGradient", "stop", "text", "tspan", "title", "desc", "style", "pattern", "marker", "filter"
}

SVG_MAX_LEN = 16000
SVG_MAX_PATHS = 256

DISALLOWED_PATTERNS = [
    r"<script\b",
    r"<foreignObject\b",
    r"<animate\b",
    r"<set\b",
    r"<animateTransform\b",
    r"on\w+\s*=",
    r"xlink:href\s*=\s*[\"']https?://",
    r"href\s*=\s*[\"']https?://",
]

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

def strip_ns(tag):
    return tag.split("}", 1)[-1] if "}" in tag else tag

def basic_svg_clean(svg):
    svg = str(svg).strip()
    svg = re.sub(r"^\s*```(?:xml|svg)?", "", svg, flags=re.IGNORECASE).strip()
    svg = re.sub(r"```\s*$", "", svg).strip()
    return svg

def extract_svg(text):
    m = SVG_REGEX.search(str(text))
    return m.group(0).strip() if m else ""

def validate_svg(svg_text):
    svg_text = basic_svg_clean(svg_text)
    if "<svg" in svg_text.lower():
        svg_text = extract_svg(svg_text)

    if not svg_text:
        return False
    if len(svg_text) > SVG_MAX_LEN:
        return False

    for pat in DISALLOWED_PATTERNS:
        if re.search(pat, svg_text, flags=re.IGNORECASE):
            return False

    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False

    if strip_ns(root.tag) != "svg":
        return False

    path_count = 0
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag not in ALLOWED_TAGS:
            return False
        if tag == "path":
            path_count += 1
        for attr in elem.attrib:
            if attr.lower().startswith("on"):
                return False

    return path_count <= SVG_MAX_PATHS

def sanitize_svg(svg_text):
    svg_text = basic_svg_clean(svg_text)
    if "<svg" in svg_text.lower():
        svg_text = extract_svg(svg_text)

    if not svg_text:
        return ""

    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return ""

    if strip_ns(root.tag) != "svg":
        return ""

    root.set("xmlns", "http://www.w3.org/2000/svg")
    root.set("width", "256")
    root.set("height", "256")
    root.set("viewBox", "0 0 256 256")

    path_count = 0
    for elem in root.iter():
        tag = strip_ns(elem.tag)
        if tag not in ALLOWED_TAGS:
            return ""
        if tag == "path":
            path_count += 1

        for attr in list(elem.attrib.keys()):
            attr_l = attr.lower()
            val_l = str(elem.attrib[attr]).lower()
            if attr_l.startswith("on"):
                del elem.attrib[attr]
            elif "http://" in val_l or "https://" in val_l:
                del elem.attrib[attr]
            elif attr_l in ("href", "xlink:href") and (val_l.startswith("http") or val_l.startswith("//")):
                del elem.attrib[attr]

    if path_count > SVG_MAX_PATHS:
        return ""

    cleaned = ET.tostring(root, encoding="unicode")
    if len(cleaned) > SVG_MAX_LEN:
        return ""
    return cleaned if validate_svg(cleaned) else ""

def normalizing_svg(svg_text):
    svg_text = sanitize_svg(svg_text)
    if not svg_text:
        return ""

    svg_text = svg_text.replace("ns0:", "")
    svg_text = svg_text.replace(":ns0", "")
    svg_text = re.sub(r'\s+xmlns:ns0="[^"]+"', "", svg_text)
    svg_text = re.sub(r"<(/?)ns\d+:", r"<\1", svg_text)
    svg_text = re.sub(r"\s+ns\d+:", " ", svg_text)

    if 'xmlns="http://www.w3.org/2000/svg"' not in svg_text:
        svg_text = svg_text.replace(
            "<svg ",
            '<svg xmlns="http://www.w3.org/2000/svg" ',
            1
        )
    return svg_text

In [ ]:
def to_prompt_svg(row):
    prompt = str(row["prompt"]).strip()
    svg = basic_svg_clean(row["svg"])

    if not prompt:
        return {"prompt": "", "svg": ""}
    if not validate_svg(svg):
        return {"prompt": "", "svg": ""}

    svg = normalizing_svg(svg)
    if not svg:
        return {"prompt": "", "svg": ""}

    return {"prompt": prompt, "svg": svg}

rows = [to_prompt_svg(row) for _, row in train_df.iterrows()]
cleaned = pd.DataFrame(rows)
cleaned = cleaned[(cleaned["prompt"] != "") & (cleaned["svg"] != "")].reset_index(drop=True)

print(cleaned.shape)
cleaned.head()

In [ ]:
cleaned["svg_len"] = cleaned["svg"].str.len()

filtered = cleaned[
    (cleaned["svg_len"] >= 500) &
    (cleaned["svg_len"] <= 8000)
].copy()

filtered["len_bucket"] = pd.qcut(filtered["svg_len"], q=8, duplicates="drop")

sampled = (
    filtered.groupby("len_bucket", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 5000), random_state=SEED))
    .reset_index(drop=True)
)

cleaned = sampled[["prompt", "svg"]].copy().reset_index(drop=True)

print(cleaned.shape)
print(cleaned["svg"].str.len().describe())

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

model.print_trainable_parameters()

In [ ]:
train_part, eval_part = train_test_split(
    clean_df,
    test_size=CONFIG["eval_size"],
    random_state=SEED,
    shuffle=True,
)

train_ds = Dataset.from_pandas(train_part.reset_index(drop=True))
eval_ds = Dataset.from_pandas(eval_part.reset_index(drop=True))

print(len(train_ds), len(eval_ds))

In [ ]:
SYSTEM_PROMPT = "Generate one valid SVG."

def format_shift(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["svg"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_text = train_ds.map(format_shift, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_shift, remove_columns=eval_ds.column_names)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=0.05,
    weight_decay=CONFIG["weight_decay"],
    fp16=False,
    bf16=True,
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    args=training_args,
)

In [ ]:
train_result = trainer.train()
print(train_result)

In [ ]:
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print("Saved to:", CONFIG["output_dir"])